In [1]:
# ICDPROCEDURES.ipynb

# ======================
# Step 1. Import libraries
# ======================
import pandas as pd
import os

base_path = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/builtdata/csv_exports"
output_path = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files"

# ======================
# Step 2. Load source data
# ======================
hosp_proc_icd = pd.read_csv(os.path.join(base_path, "hosp_procedures_icd.csv"))
hosp_d_icd = pd.read_csv(os.path.join(base_path, "hosp_d_icd_procedures.csv"))

print("hosp_procedures_icd:", hosp_proc_icd.shape)
print("hosp_d_icd_procedures:", hosp_d_icd.shape)

# ======================
# Step 3. Merge hospital ICD procedures with descriptions
# ======================
proc_merged = hosp_proc_icd.merge(
    hosp_d_icd,
    on=["icd_code", "icd_version"],
    how="left"
)

print("Merged:", proc_merged.shape)

# ======================
# Step 4. Build final ICDPROCEDURES table
# ======================
icd_proc_final = pd.DataFrame({
    "pat_id": proc_merged["subject_id"],
    "csn": proc_merged["hadm_id"],
    "icd9_procedure_code": proc_merged.apply(
        lambda x: x["icd_code"] if x["icd_version"] == 9 else "NOT AVAILABLE", axis=1
    ),
    "icd10_procedure_code": proc_merged.apply(
        lambda x: x["icd_code"] if x["icd_version"] == 10 else "NOT AVAILABLE", axis=1
    ),
    "procedure_desc": proc_merged["long_title"].fillna("UNKNOWN"),
    "procedure_date": proc_merged["chartdate"]
})

print("✅ Final ICDPROCEDURES:", icd_proc_final.shape)
display(icd_proc_final.head(10))

# ======================
# Step 5. Save output
# ======================
out_file = os.path.join(output_path, "ICD_PROCEDURES.csv")
icd_proc_final.to_csv(out_file, index=False)
print(f"ICDPROCEDURES file saved to {out_file}")

hosp_procedures_icd: (859655, 6)
hosp_d_icd_procedures: (86423, 3)
Merged: (859655, 7)
✅ Final ICDPROCEDURES: (859655, 6)


,pat_id,csn,icd9_procedure_code,icd10_procedure_code,procedure_desc,procedure_date
0,10000032,22595853,5491,NOT AVAILABLE,UNKNOWN,2180-05-07
1,10000032,22841357,5491,NOT AVAILABLE,UNKNOWN,2180-06-27
2,10000032,25742920,5491,NOT AVAILABLE,UNKNOWN,2180-08-06
3,10000068,25022803,8938,NOT AVAILABLE,UNKNOWN,2160-03-03
4,10000117,27988844,NOT AVAILABLE,0QS734Z,Reposition Left Upper Femur with Internal Fixa...,2183-09-19
5,10000280,25852320,8938,NOT AVAILABLE,UNKNOWN,2151-03-18
6,10000560,28979390,5551,NOT AVAILABLE,UNKNOWN,2189-10-16
7,10000635,26134563,3734,NOT AVAILABLE,UNKNOWN,2136-06-19
8,10000635,26134563,3728,NOT AVAILABLE,UNKNOWN,2136-06-19
9,10000635,26134563,3727,NOT AVAILABLE,UNKNOWN,2136-06-19


ICDPROCEDURES file saved to /hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files/ICD_PROCEDURES.csv
